# Optimization for AI Project - Deiana

For this project, we will explore and try to design a GA algorithm to automatically synthesize FIR filters according to a desired frequency response. We will start with a baseline simplified proof of concept, and then we will take inspiration from a couple of papers to produce more useful and realistic results.

A digital FIR filter is characterized by the following transfer function: $H(s)=\sum^{N}_{n=0}a_nz^{-n}$, where $N$ is the order of the filter, and $a_n$ are
defined as the filter coefficients.

As we can see, an individual in this case is simply a finite vector of real values, which allows us to inherit all the literature about real-valued GAs. In reality,
digital FIR filters are designed to be implemented in hardware. This presents us with a couple of challenges, namely that hardware costs increase with the order of the filter and the way it's implemented.
As we will later see, a particular ternary encoding called CSD is preferred, as it allows to automatically synthesize a filter from the provided genome and saves up on circuit complexity.
 This, in turn, means that the real-valued GAs, which are quite simple to implement in software, are not desirable for real-world FIR filter design. Nonetheless, precisely because of its simplicity,
 we will first implement a real-valued version of the GA as a baseline, and compare it wrt. execution speed, convergence speed, and fitness with the algorithms proposed in the papers.

## Real-valued GA baseline

Before we implement the techniques proposed in the papers, it could be interesting to create a simple proof of concept at a higher level. We will implement the classic GA structure, and encode the individuals as simple vectors of real numbers. This will allow us to reuse much of the code when implementing the papers.

The GA will be implemented as follows:
- Chromosome: $x=(x_1,…,x_n)$, where $x_i \in [-1, 1] \subset \mathbb{R}, \forall i \in {0, ..., n}$
- Init: Uniform sampling
- Mutation: Gaussian
- Crossover: Two-point crossover, Discrete recombination
- Selection: Roulette Wheel, Tournament
- Fitness: Minimax, LMS, MSE
- Elitism: 5% of the population [OPTIONAL]

We will use the signal library from Scipy for calculating the frequency response of each individual, together with numpy. For everything else, standard python is enough to implement the GA.

### Implementation

In [224]:
import random, bisect
import numpy as np
from scipy.signal import freqz

In [225]:
def sample_individual(order):
    individual = []
    for _ in range(order):
        individual.append(random.choice([random.uniform(-1, 1), 0]))
    return individual

In [226]:
def init_pop(order, n_pop):
    return [sample_individual(order) for _ in range(n_pop)]

In [227]:
def mut(individual, p=0.01):
    mutated = []
    for coeff in individual:
        if random.random() < p:
            mutated.append(coeff + random.gauss(sigma=1-abs(coeff))) # We want to remain in the [-1, 1] range
        else:
            mutated.append(coeff)
    return mutated

In [228]:
def twopoint_cross(parent1, parent2):
    indexes = random.sample(range(1, len(parent1)-1), 2)
    indexes.sort()
    return parent1[:indexes[0]] + parent2[indexes[0]:indexes[1]] + parent1[indexes[1]:]

def recomb_cross(parent1, parent2):
    child = []
    for p, q in zip(parent1, parent2):
        a = random.uniform(0, 1)
        child.append(a*p + (1-a)*q)
    return child

def cross(parent1, parent2, strategy=twopoint_cross):
    return strategy(parent1, parent2)

In [229]:
def minimax_error(individual, target):
    wi, Hi = freqz(individual)
    Ht = [target(w) for w in wi]
    error = max(np.abs(Ht - Hi))
    return error

def lms_error(individual, target):
    wi, Hi = freqz(individual)
    Ht = [target(w) for w in wi]
    error = np.sqrt(np.sum(np.power(np.abs(Ht - Hi), 2)))
    return error

def mse(individual, target):
    wi, Hi = freqz(individual)
    Ht = [target(w) for w in wi]
    error = np.average(np.sum(np.power(np.abs(Ht - Hi), 2)))
    return error

def minimax_fit(individual, target):
    return -minimax_error(individual, target)

def lms_fit(individual, target):
    return -lms_error(individual, target)

def mse_fit(individual, target):
    return -mse(individual, target)

In [230]:
def roulette_selection(pop, n_pop, fitness, target):
    # Compute fitness values once
    fvals = [fitness(ind, target) for ind in pop]
    tot = sum(fvals)
    # Build cumulative distribution
    cum = []
    s = 0.0
    for fv in fvals:
        s += fv
        cum.append(s)
    newpop = []
    for _ in range(n_pop):
        r = random.random() * tot  # r ∈ [0, tot)
        idx = bisect.bisect_left(cum, r)
        newpop.append(pop[idx])
    return newpop

def tournament_selection(pop, n_pop, fitness, target, k=3):
    newpop = []
    while len(newpop) < n_pop:
        contestants = random.choices(population=pop, k=k)
        contestants.sort(key=lambda x : fitness(x, target), reverse=True)
        newpop.append(contestants[0])
    return newpop

def selection(pop, n_pop, fitness, target, strategy=roulette_selection):
    return strategy(pop, n_pop, fitness, target)

In [231]:
def GA(n_pop, generations, order, target, fitness, elitism=False):
    best = []
    elites_n = int(n_pop / 100 * 5) if elitism else 0
    # Population initialization
    pop = init_pop(order, n_pop)
    for _ in range(generations):
        pop.sort(key=lambda x : fitness(x, target), reverse=True)
        elites = pop[:elites_n]
        best.append(pop[0])
        for _ in range(n_pop):
            parent1 = pop[random.randint(0, len(pop)-1)]
            parent2 = pop[random.randint(0, len(pop)-1)]
            child1 = mut(cross(parent1, parent2))
            child2 = mut(cross(parent2, parent1))
            pop.append(child1)
            pop.append(child2)
        pop = selection(pop, n_pop-elites_n, fitness, target)
        pop.extend(elites)
    return best

### Testing "classic" FIR filters

In [232]:
import matplotlib.pyplot as plt

In [233]:
n_pop = 100
generations = 100
order = 8

#### 0. Ideal all-pass filter

In [234]:
def target(w):
    return 1

In [235]:
best_minimax = GA(n_pop, generations, order, target, minimax_fit, elitism=True)
best_lms = GA(n_pop, generations, order, target, lms_fit, elitism=True)
best_mse = GA(n_pop, generations, order, target, mse_fit, elitism=True)

/tmp/ipykernel_23356/3271014338.py:20: RuntimeWarning: divide by zero encountered in scalar divide
  return 1/minimax_error(individual, target)
/tmp/ipykernel_23356/3271014338.py:23: RuntimeWarning: divide by zero encountered in scalar divide
  return 1/lms_error(individual, target)


KeyboardInterrupt: 

In [ ]:
w1, H1 = freqz(best_minimax[-1])
w2, H2 = freqz(best_lms[-1])
w3, H3 = freqz(best_mse[-1])
wt = w1
Ht = np.array([target(w) for w in wt])  # target returns complex H(ω)

mag = {
    'Minimax': np.abs(H1),
    'LMS':     np.abs(H2),
    'MSE':     np.abs(H3),
    'Target':  np.abs(Ht),
}

phase = {
    'Minimax': np.unwrap(np.angle(H1)),
    'LMS':     np.unwrap(np.angle(H2)),
    'MSE':     np.unwrap(np.angle(H3)),
    'Target':  np.unwrap(np.angle(Ht)),
}

colors = {
    'Minimax': 'blue',
    'LMS':     'green',
    'MSE':     'magenta',
    'Target':  'red',
}

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax_mag.scatter(w1, mag['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_mag.scatter(w2, mag['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_mag.scatter(w3, mag['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_mag.scatter(wt, mag['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_mag.set_ylabel(r'$|H(e^{j\omega})|$')
ax_mag.set_title('Frequency Response — Magnitude & Phase')
ax_mag.grid(True, alpha=0.3)
ax_mag.legend(loc='best')

ax_phase.scatter(w1, phase['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_phase.scatter(w2, phase['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_phase.scatter(w3, phase['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_phase.scatter(wt, phase['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_phase.set_xlabel(r'$\omega$ (rad/sample)')
ax_phase.set_ylabel(r'$\angle H(e^{j\omega})$ (rad)')
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


#### 1. Ideal high-pass filter

In [ ]:
def target(w):
    return 1 if w >= 1.5 else 0

In [ ]:
best_minimax = GA(n_pop, generations, order, target, minimax_fit, elitism=True)
best_lms = GA(n_pop, generations, order, target, lms_fit, elitism=True)
best_mse = GA(n_pop, generations, order, target, mse_fit, elitism=True)

In [ ]:
w1, H1 = freqz(best_minimax[-1])
w2, H2 = freqz(best_lms[-1])
w3, H3 = freqz(best_mse[-1])
wt = w1
Ht = np.array([target(w) for w in wt])  # target returns complex H(ω)

mag = {
    'Minimax': np.abs(H1),
    'LMS':     np.abs(H2),
    'MSE':     np.abs(H3),
    'Target':  np.abs(Ht),
}

phase = {
    'Minimax': np.unwrap(np.angle(H1)),
    'LMS':     np.unwrap(np.angle(H2)),
    'MSE':     np.unwrap(np.angle(H3)),
    'Target':  np.unwrap(np.angle(Ht)),
}

colors = {
    'Minimax': 'blue',
    'LMS':     'green',
    'MSE':     'magenta',
    'Target':  'red',
}

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax_mag.scatter(w1, mag['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_mag.scatter(w2, mag['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_mag.scatter(w3, mag['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_mag.scatter(wt, mag['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_mag.set_ylabel(r'$|H(e^{j\omega})|$')
ax_mag.set_title('Frequency Response — Magnitude & Phase')
ax_mag.grid(True, alpha=0.3)
ax_mag.legend(loc='best')

ax_phase.scatter(w1, phase['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_phase.scatter(w2, phase['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_phase.scatter(w3, phase['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_phase.scatter(wt, phase['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_phase.set_xlabel(r'$\omega$ (rad/sample)')
ax_phase.set_ylabel(r'$\angle H(e^{j\omega})$ (rad)')
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


#### 2. Ideal band-pass filter

In [ ]:
def target(w):
    return 0 if w >= 2 or w <= 1 else 1

In [ ]:
best_minimax = GA(n_pop, generations, order, target, minimax_fit, elitism=True)
best_lms = GA(n_pop, generations, order, target, lms_fit, elitism=True)
best_mse = GA(n_pop, generations, order, target, mse_fit, elitism=True)

In [ ]:
w1, H1 = freqz(best_minimax[-1])
w2, H2 = freqz(best_lms[-1])
w3, H3 = freqz(best_mse[-1])
wt = w1
Ht = np.array([target(w) for w in wt])  # target returns complex H(ω)

mag = {
    'Minimax': np.abs(H1),
    'LMS':     np.abs(H2),
    'MSE':     np.abs(H3),
    'Target':  np.abs(Ht),
}

phase = {
    'Minimax': np.unwrap(np.angle(H1)),
    'LMS':     np.unwrap(np.angle(H2)),
    'MSE':     np.unwrap(np.angle(H3)),
    'Target':  np.unwrap(np.angle(Ht)),
}

colors = {
    'Minimax': 'blue',
    'LMS':     'green',
    'MSE':     'magenta',
    'Target':  'red',
}

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax_mag.scatter(w1, mag['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_mag.scatter(w2, mag['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_mag.scatter(w3, mag['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_mag.scatter(wt, mag['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_mag.set_ylabel(r'$|H(e^{j\omega})|$')
ax_mag.set_title('Frequency Response — Magnitude & Phase')
ax_mag.grid(True, alpha=0.3)
ax_mag.legend(loc='best')

ax_phase.scatter(w1, phase['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_phase.scatter(w2, phase['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_phase.scatter(w3, phase['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_phase.scatter(wt, phase['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_phase.set_xlabel(r'$\omega$ (rad/sample)')
ax_phase.set_ylabel(r'$\angle H(e^{j\omega})$ (rad)')
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


#### 3. Ideal differentiator

In [ ]:
def target(w):
    return w

In [ ]:
best_minimax = GA(n_pop, generations, order, target, minimax_fit, elitism=True)
best_lms = GA(n_pop, generations, order, target, lms_fit, elitism=True)
best_mse = GA(n_pop, generations, order, target, mse_fit, elitism=True)

In [ ]:
w1, H1 = freqz(best_minimax[-1])
w2, H2 = freqz(best_lms[-1])
w3, H3 = freqz(best_mse[-1])
wt = w1
Ht = np.array([target(w) for w in wt])  # target returns complex H(ω)

mag = {
    'Minimax': np.abs(H1),
    'LMS':     np.abs(H2),
    'MSE':     np.abs(H3),
    'Target':  np.abs(Ht),
}

phase = {
    'Minimax': np.unwrap(np.angle(H1)),
    'LMS':     np.unwrap(np.angle(H2)),
    'MSE':     np.unwrap(np.angle(H3)),
    'Target':  np.unwrap(np.angle(Ht)),
}

colors = {
    'Minimax': 'blue',
    'LMS':     'green',
    'MSE':     'magenta',
    'Target':  'red',
}

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax_mag.scatter(w1, mag['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_mag.scatter(w2, mag['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_mag.scatter(w3, mag['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_mag.scatter(wt, mag['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_mag.set_ylabel(r'$|H(e^{j\omega})|$')
ax_mag.set_title('Frequency Response — Magnitude & Phase')
ax_mag.grid(True, alpha=0.3)
ax_mag.legend(loc='best')

ax_phase.scatter(w1, phase['Minimax'], c=colors['Minimax'], marker='.', label='Minimax')
ax_phase.scatter(w2, phase['LMS'],     c=colors['LMS'],     marker='.', label='LMS')
ax_phase.scatter(w3, phase['MSE'],     c=colors['MSE'],     marker='.', label='MSE')
ax_phase.scatter(wt, phase['Target'],  c=colors['Target'],  marker='.', label='Target')

ax_phase.set_xlabel(r'$\omega$ (rad/sample)')
ax_phase.set_ylabel(r'$\angle H(e^{j\omega})$ (rad)')
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
